# Demo: a single N-AFC trial

An **N-alternative forced choice (N-AFC)** trial plays *N* sound intervals and
asks the participant to pick one of them — e.g. *"which interval is the odd one
out?"*. It is the building block of many discrimination experiments.

This notebook runs **one** N-AFC trial with `whispy.ui.NAFC` and walks through
the three things an experimenter needs to know:

1. how to describe a trial (the `screen` dict),
2. how the window behaves for the participant,
3. what the result looks like.

Once you understand a single trial, you run many of them by building a
`screen` per trial — either with a fixed design (`whispy.ExperimentScheduler`)
or adaptively (`whispy.Staircase`, see `staircase_n_afc.ipynb`).

If you haven't installed the project yet, run this from the repo root:
```bash
pip install -e .
```

In [ ]:
import whispy
from pathlib import Path
from whispy.interfaces import SoundDevice
from whispy.ui import NAFC

## 1. Stimuli & playback handler

Sounds are played by a `StimuliHandler`. The default `SoundDevice` handler maps
**stimulus ids** to WAV files via `configs/stimuli.yml` and plays them from a
directory of audio files. For this demo we use the bundled example stimuli
(ids `1..4`).

In a real experiment, give your stimuli descriptive ids — the id of the
interval the participant picks is saved in the results.

In [ ]:
config_dir = Path('..') / 'configs'
stimuli_dir = Path('demo_stimuli/n_afc')

stimuli_handler = SoundDevice(config_dir / 'stimuli.yml', stimuli_dir, loop=False)
print('available stimulus ids:', list(stimuli_handler.stimuli.keys()))

## 2. Describe the trial: the `screen` dict

`NAFC` is told what to present with a single `screen` dictionary. The keys that
matter for a trial:

| key | meaning |
|-----|---------|
| `test` | list of stimulus ids, one per interval/button. Its length is *N*. |
| `correct` | the stimulus id that counts as the correct answer (optional). |
| `attribute` | name of an entry in `configs/attributes.yml`; its `task` text is shown as the on-screen prompt. |
| `trial_id`, `block`, `section`, `block_name`, `section_name` | metadata copied straight into the results. |

Below we build a **3-AFC "odd one out"** trial: two intervals play a fixed
*standard* (id `1`) and one plays a different *target* (id `4`); the target is
the correct answer. The buttons are shuffled by `NAFC`, so the participant
can't learn a fixed position (controlled by `shuffle_choices` in
`configs/n_afc.yml`).

In [ ]:
standard = 1
target = 4

screen = {
    'test': [standard, standard, target],  # 3 intervals -> 3-AFC
    'correct': target,
    'attribute': 'difference',              # prompt text comes from attributes.yml
    'trial_id': 0,
    'block': 0,
    'section': 0,
    'block_name': 'Demo',
    'section_name': '3-AFC',
}
screen

## 3. Run the trial

Running the next cell opens the N-AFC window. As the participant:

- click a numbered button to **hear** that interval (click again / others to compare),
- select the interval you think is the odd one out,
- press **"Submit choice"** to confirm.

With `blocking=True`, the cell waits until you submit, then returns. The window
stays open afterwards so you can read the result; we close it explicitly in
step 5. (The wording of the prompt and submit hint live in
`configs/attributes.yml` and `configs/n_afc.yml`.)

In [ ]:
naf = NAFC(screen=screen, stimuli_handler=stimuli_handler, blocking=True)

## 4. Read the result

`get_results()` returns a one-row `pandas.DataFrame` for the trial:

- `choices` — the stimulus ids in the order shown to the participant,
- `selected` — the id the participant picked,
- `correct` / `correct_bool` — the correct id and whether the pick matched it,
- `rt` — reaction time (seconds) from window open to submit,
- plus the `trial_id` / `block` / `section` metadata from the `screen`.

Run many trials by collecting one such row each and concatenating them.

In [ ]:
results = naf.get_results()
results

## 5. Close the window

Submitting only releases the trial; the window is kept open so a multi-trial
loop can reuse it. Close it explicitly when you're done.

In [ ]:
naf.close()

## Where to go next

- **Customize the look/wording**: `configs/n_afc.yml` (window size, submit hint,
  button label, `n_choices`, `shuffle_choices`) and `configs/attributes.yml`
  (the prompt text). Colors/fonts are shared across all UIs via
  `configs/design.yml`.
- **Run a fixed set of trials**: drive `NAFC` from a
  `whispy.ExperimentScheduler` (method of constant stimuli).
- **Run an adaptive threshold experiment**: see `staircase_n_afc.ipynb`, which
  reuses one window across many trials.